In [2]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt


In [7]:
columns = [
    "unit_id" , 
    "cycle",
    "setting_1",
    "setting_2",
    "setting_3",
]

sensor_columns = [f"sensor_{i}" for i in range(1,22)]

columns.extend(sensor_columns)

In [11]:
train_df = pd.read_csv(
    "../data/train_FD001.txt",
    sep=r"\s+",
    header = None,
    names = columns
)

train_df.head(5)
    
    

,unit_id,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [12]:
#create RUL 
#RUL = Maximum cycle of engine - current cycle 

max_cycle = train_df.groupby("unit_id")["cycle"].max()

train_df["RUL"] = train_df.apply(
    lambda row: max_cycle[row["unit_id"]] - row["cycle"],
    axis =1
)

In [13]:
train_df[["unit_id" , "cycle" , "RUL"]].head(10)

,unit_id,cycle,RUL
0,1,1,191.0
1,1,2,190.0
2,1,3,189.0
3,1,4,188.0
4,1,5,187.0
5,1,6,186.0
6,1,7,185.0
7,1,8,184.0
8,1,9,183.0
9,1,10,182.0


In [15]:
selected_sensors = sensor_columns.copy()

print("Number of Sensor :", len(selected_sensors))
print(selected_sensors)

Number of Sensor : 21
['sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21']


In [20]:
#Rolling mean
#a rolling mean smooths the sensor reading and helps up see thde underlying trend
window = 5

for sensor in selected_sensors:
    train_df[f"{sensor}_rolling_mean"] = (
        train_df
        .groupby("unit_id")[sensor]
        .transform(lambda x: x.rolling(window=window).mean()))

In [21]:
train_df.head()

,unit_id,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12_rolling_mean,sensor_13_rolling_mean,sensor_14_rolling_mean,sensor_15_rolling_mean,sensor_16_rolling_mean,sensor_17_rolling_mean,sensor_18_rolling_mean,sensor_19_rolling_mean,sensor_20_rolling_mean,sensor_21_rolling_mean
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.282,2388.048,8134.194,8.41334,0.03,391.8,2388.0,100.0,38.958,23.39302


In [22]:
for sensor in selected_sensors:
    train_df[f"{sensor}_rolling_std"] = (
        train_df
        .groupby("unit_id")[sensor]
        .transform(lambda x: x.rolling(window=window).std())
    )